(course-biophysics-pbc-geometry)=

# Path D - Module 40: PBC Geometry & Conventions

Membrane simulations are highly sensitive to PBC. In this module, you will learn to manage the semi-isotropic box scaling of a lipid bilayer.

In [1]:
import numpy as np
import molsysmt as msm
from molsysmt import systems

molsys = [systems['POPC membrane']['popc_membrane.psf'],
          systems['POPC membrane']['popc_membrane.dcd']]
structure_indices = [0, 2, 1]
box = msm.get(molsys, element='system', structure_indices=structure_indices, box=True)
print(f"Membrane boxes in requested structure order:\n{box}")


Membrane boxes in requested structure order:
[[[9.573329 0.0 0.0]  [0.0 9.573329 0.0]  [0.0 0.0 8.286052]] [[9.56081 0.0 0.0]  [0.0 9.56081 0.0]  [0.0 0.0 8.317084]] [[9.559361 0.0 0.0]  [0.0 9.559361 0.0]  [0.0 0.0 8.305899]]] nanometer


### Box parameters and volume

Use the registered membrane PSF/DCD pair. Explicit, nonconsecutive structure
indices keep the requested order and avoid loading a complete coordinate
ensemble merely to inspect these boxes. MolSysMT reports lengths and volume in
your configured units. This bilayer uses orthogonal boxes; a triclinic cell
requires its actual angles rather than orthogonal defaults.


In [2]:
lengths, angles = msm.pbc.get_lengths_and_angles_from_box(box)
volume = msm.pbc.get_volume_from_lengths_and_angles(lengths, angles)
print(f"Box lengths: {lengths}")
print(f"Box angles: {angles}")
print(f"Box volume: {volume}")
reconstructed = msm.pbc.get_box_from_lengths_and_angles(lengths)
assert reconstructed.shape == (len(structure_indices), 3, 3)
np.testing.assert_allclose(msm.pyunitwizard.get_value(reconstructed, to_unit='nm'),
    msm.pyunitwizard.get_value(box, to_unit='nm'), atol=1e-6)


Box lengths: [[9.573329 9.573329 8.286052] [9.56081 9.56081 8.317084] [9.559361 9.559361 8.305899]] nanometer
Box angles: [[1.570796 1.570796 1.570796] [1.570796 1.570796 1.570796] [1.570796 1.570796 1.570796]] radian
Box volume: [759.4052985151505 760.2570620624418 759.0045354216571] nanometer ** 3


### Inspecting a box through a view

The experimental [MolSysViewer tools addon](../../user/tools/molsysviewer_tools.md)
keeps these scientific inputs explicit under `view.addons.molsysmt.pbc`.
Only `has_pbc()` binds the current molecular system; constructing a box does not
assign it. Use `tools.basic.get(box=True, structure_indices=...)` before querying
selected boxes. Native unit policy and mean-angle shape labels still apply;
the native box factory supports its documented orthogonal default, and the
bound tool forwards that choice without assigning the resulting matrix.
The coordinate editors preserve native independent/in-place modes and share
managed scene synchronization. Centered in-place wrapping publishes to the original system under the native
selected-structure contract. Consumer admission is qualified against a specific
provider source; consult the linked addon guide for its current status.
Unwrap follows
the supplied structure order rather than reconstructing covalent components.
